In [1]:
:set -XDeriveFunctor
:set -XDeriveFoldable
:set -XDeriveTraversable
:set -XScopedTypeVariables
:set -XInstanceSigs
:set -XTupleSections
import Data.Foldable (fold, foldMap', toList, maximumBy, minimumBy, for_, traverse_, asum, sequenceA_)
import Data.Traversable (mapAccumL, mapAccumR, for)
import Data.List (intercalate, foldl', sortBy, nub, isPrefixOf)
import Data.Ord (comparing, Down(..))
import Data.Maybe (catMaybes, mapMaybe, fromMaybe, listToMaybe)
import Data.Char (toUpper, toLower, isAlpha, isDigit, digitToInt)
import Control.Monad (when, forM_, forM, replicateM)
import Control.Monad.State (State, get, put, execState, evalState, runState)
import Control.Monad.Writer (Writer, tell, execWriter, runWriter)
import Data.Map.Strict (Map)
import qualified Data.Map.Strict as Map
import Data.Set (Set)
import qualified Data.Set as Set
import Data.IORef

-- Вспомогательные функции
putHeader :: String -> IO ()
putHeader s = do
  putStrLn ""
  putStrLn $ replicate (length s + 8) '='
  putStrLn $ "=== " ++ s ++ " ==="
  putStrLn $ replicate (length s + 8) '='

# 🌀 Foldable & Traversable в Haskell

> **Foldable** — это структура, из которой можно «свернуть» значения в единый результат.  
> **Traversable** — это структура, через которую можно «пройтись» с эффектами, сохраняя форму.

Каждый раздел описан по схеме:

- 📐 **Теория** + интуиция «на пальцах»
- ⭐ **Класс типов**: методы, законы, минимальная реализация
- 🛠️ **Примеры** — от простого к сложному (минимум 3)
- 💡 **Ловушки** — контринтуитивные моменты
- 🔭 **Категорная точка зрения**

---

📌 Содержание

| # | Тема | Суть |
|---|------|------|
| 1 | `Foldable` — основы | `foldr`, `foldl`, лень, законы |
| 2 | `foldMap` и моноиды | Сердце `Foldable`, производительность |
| 3 | Встроенные функции `Foldable` | `sum`, `any`, `toList`, `maximumBy` и др. |
| 4 | `Foldable` для своих типов | Деревья, розовые деревья, графы |
| 5 | `Traversable` — основы | `traverse`, `sequenceA`, законы |
| 6 | `Traversable` для своих типов | Tree, Rose, пользовательские контейнеры |
| 7 | Аппликативные эффекты | IO, Maybe, Either, State, Writer |
| 8 | `mapAccumL` / `mapAccumR` | Аккумулятор с сохранением формы |
| 9 | Продвинутые паттерны | `for`, `forM`, `sequenceA`, композиция |
| 10 | Производительность | `foldl'`, `toList`, строгость, `Builder` |
| 11 | 🪞 Дуальность и категорная картина | Алгебры, naturality, связь с линзами |

> **📦 Зависимости**
> **Пакеты:** `base`, `containers`, `mtl`
> **Расширения:** `DeriveFoldable` — deriving Foldable ([→](Extensions.ipynb#deriving)) · `DeriveFunctor` — deriving Functor ([→](Extensions.ipynb#deriving)) · `DeriveTraversable` — deriving Traversable ([→](Extensions.ipynb#deriving)) · `InstanceSigs` — сигнатуры методов прямо в instance ([→](Extensions.ipynb#instancesigs)) · `ScopedTypeVariables` — переменные типа из сигнатуры видны в теле ([→](Extensions.ipynb#scopedtypevariables)) · `TupleSections` — частичное применение конструктора кортежа: (,5) ([→](Extensions.ipynb#tuplesections))


---
# 1️⃣ `Foldable` — Основы

## 📐 Теория

`Foldable` — класс типов для контейнеров, элементы которых можно свернуть в одно значение.

```haskell
class Foldable t where
  -- Минимальная реализация: foldr ИЛИ foldMap
  foldr   :: (a -> b -> b) -> b -> t a -> b
  foldMap :: Monoid m => (a -> m) -> t a -> m

  -- Производные методы (по умолчанию реализованы через foldr/foldMap)
  foldl   :: (b -> a -> b) -> b -> t a -> b
  foldr'  :: (a -> b -> b) -> b -> t a -> b  -- строгий
  foldl'  :: (b -> a -> b) -> b -> t a -> b  -- строгий
  fold    :: Monoid m => t m -> m
  toList  :: t a -> [a]
  null    :: t a -> Bool
  length  :: t a -> Int
  elem    :: Eq a => a -> t a -> Bool
  maximum :: Ord a => t a -> a
  minimum :: Ord a => t a -> a
  sum     :: Num a => t a -> a
  product :: Num a => t a -> a
```

**Интуиция:** думай о `foldr f z` как о замене каждого конструктора списка `(:)` на `f`, а `[]` на `z`:

```
[a, b, c]  ≡  a : (b : (c : []))
             ↓ foldr f z
             f a (f b (f c z))
```

**`foldr` vs `foldl`:**
```
foldr f z [a,b,c] = f a (f b (f c z))    -- правая скобка
foldl f z [a,b,c] = f (f (f z a) b) c   -- левая скобка
```

## ⭐ Законы

```haskell
-- Согласованность foldr и foldMap:
foldr f z t = appEndo (foldMap (Endo . f) t) z

-- Согласованность toList и foldMap:
foldMap f = fold . fmap f  -- только при Functor
```

Формальных законов у Foldable немного — класс намеренно гибкий.

In [2]:
-- Пример 1: foldr — визуализация шагов
demo_foldr :: IO ()
demo_foldr = do
  putHeader "foldr — шаги"
  let xs = [1,2,3,4,5] :: [Int]

  -- Базовые операции
  putStrLn $ "foldr (+) 0 [1..5]  = " ++ show (foldr (+) 0 xs)
  putStrLn $ "foldr (*) 1 [1..5]  = " ++ show (foldr (*) 1 xs)
  putStrLn $ "foldr (:) [] [1..5] = " ++ show (foldr (:) [] xs)  -- id
  putStrLn $ "foldr (\\x acc -> acc ++ [x]) [] [1..5] = " ++ show (foldr (\x acc -> acc ++ [x]) [] xs) -- reverse

  -- Визуализация шагов
  let showStepR = foldr (\x acc -> "(" ++ show x ++ " + " ++ acc ++ ")") "0" (map show xs)
  putStrLn $ "\nfoldr (+) 0 [1..5] развёртка:"
  putStrLn $ "  " ++ showStepR

  let showStepL = foldl (\acc x -> "(" ++ acc ++ " + " ++ show x ++ ")") "0" (map show xs)
  putStrLn $ "foldl (+) 0 [1..5] развёртка:"
  putStrLn $ "  " ++ showStepL

  -- foldr на бесконечных списках — работает благодаря лени!
  putHeader "foldr на бесконечных списках"
  let firstGt100 = foldr (\x acc -> if x > 100 then Just x else acc) Nothing [1..200]
  putStrLn $ "Первый элемент > 100 в [1..]: " ++ show firstGt100

  let take5 = foldr (\x acc -> if length acc >= 5 then acc else x:acc) [] [1..100]
  putStrLn $ "Первые 5 элементов через foldr: " ++ show take5

demo_foldr

Line 2: Use camelCase
Found:
demo_foldr :: IO ()
Why not:
demoFoldr :: IO ()Line 3: Use camelCase
Found:
demo_foldr = ...
Why not:
demoFoldr = ...Line 8: Use sum
Found:
foldr (+) 0
Why not:
sumLine 9: Use product
Found:
foldr (*) 1
Why not:
productLine 14: Fuse foldr/map
Found:
foldr
  (\ x acc -> "(" ++ show x ++ " + " ++ acc ++ ")") "0" (map show xs)
Why not:
foldr
  ((\ x acc -> "(" ++ show x ++ " + " ++ acc ++ ")") . show) "0" xsLine 15: Redundant $
Found:
putStrLn $ "\nfoldr (+) 0 [1..5] развёртка:"
Why not:
putStrLn "\nfoldr (+) 0 [1..5] развёртка:"Line 19: Redundant $
Found:
putStrLn $ "foldl (+) 0 [1..5] развёртка:"
Why not:
putStrLn "foldl (+) 0 [1..5] развёртка:"


=== foldr — шаги ===
foldr (+) 0 [1..5]  = 15
foldr (*) 1 [1..5]  = 120
foldr (:) [] [1..5] = [1,2,3,4,5]
foldr (\x acc -> acc ++ [x]) [] [1..5] = [5,4,3,2,1]

foldr (+) 0 [1..5] развёртка:
  ("1" + ("2" + ("3" + ("4" + ("5" + 0)))))
foldl (+) 0 [1..5] развёртка:
  (((((0 + "1") + "2") + "3") + "4") + "5")

=== foldr на бесконечных списках ===
Первый элемент > 100 в [1..]: Just 101
Первые 5 элементов через foldr: [96,97,98,99,100]

In [3]:
-- Пример 2: foldl' — строгая свёртка для агрегаций
-- scanl': промежуточные аккумуляторы
myScanl' :: (b -> a -> b) -> b -> [a] -> [b]
myScanl' f z [] = [z]
myScanl' f z (x:xs) = let z' = f z x in z' `seq` (z : myScanl' f z' xs)

demo_strict_fold :: IO ()
demo_strict_fold = do
  putHeader "foldl' — строгий аккумулятор"
  let n = 10000 :: Int
  let xs = [1..n]

  let s = foldl' (+) 0 xs
  putStrLn $ "foldl' (+) 0 [1..10000] = " ++ show s

  let wordList = ["apple", "banana", "apple", "cherry", "banana", "apple"]
  let freq = foldl' (\acc w -> Map.insertWith (+) w 1 acc) Map.empty wordList
  putStrLn $ "\nЧастота слов: " ++ show freq

  let (total, count) = foldl' (\(s,n) x -> (s+x, n+1)) (0,0) ([1..10] :: [Double])
  putStrLn $ "\nСреднее [1..10] = " ++ show (total / count)

  let running = myScanl' (+) 0 [1..5 :: Int]
  putStrLn $ "\nscanl' (+) 0 [1..5] = " ++ show running

demo_strict_fold

Line 7: Use camelCase
Found:
demo_strict_fold :: IO ()
Why not:
demoStrictFold :: IO ()Line 8: Use camelCase
Found:
demo_strict_fold = ...
Why not:
demoStrictFold = ...


=== foldl' — строгий аккумулятор ===
foldl' (+) 0 [1..10000] = 50005000

Частота слов: fromList [("apple",3),("banana",2),("cherry",1)]

Среднее [1..10] = 5.5

scanl' (+) 0 [1..5] = [0,1,3,6,10,15]

In [4]:
-- Пример 3: Foldable для своего дерева (три порядка обхода)
data BTree a = BLeaf | BNode (BTree a) a (BTree a)
  deriving (Show, Functor)

-- in-order: левый → корень → правый
instance Foldable BTree where
  foldr _ z BLeaf         = z
  foldr f z (BNode l x r) = foldr f (f x (foldr f z r)) l

-- Вспомогательные функции обхода
preOrder :: BTree a -> [a]
preOrder BLeaf         = []
preOrder (BNode l x r) = x : preOrder l ++ preOrder r

postOrder :: BTree a -> [a]
postOrder BLeaf         = []
postOrder (BNode l x r) = postOrder l ++ postOrder r ++ [x]

-- Пример дерева:
--       5
--      / \
--     3   8
--    / \ / \
--   1  4 7  9
myTree :: BTree Int
myTree = BNode
  (BNode (BNode BLeaf 1 BLeaf) 3 (BNode BLeaf 4 BLeaf))
  5
  (BNode (BNode BLeaf 7 BLeaf) 8 (BNode BLeaf 9 BLeaf))

demo_tree :: IO ()
demo_tree = do
  putHeader "BTree — три порядка обхода"
  putStrLn $ "in-order  (toList): " ++ show (toList myTree)
  putStrLn $ "pre-order:          " ++ show (preOrder myTree)
  putStrLn $ "post-order:         " ++ show (postOrder myTree)
  putStrLn ""
  putStrLn $ "sum:     " ++ show (sum myTree)
  putStrLn $ "product: " ++ show (product myTree)
  putStrLn $ "maximum: " ++ show (maximum myTree)
  putStrLn $ "minimum: " ++ show (minimum myTree)
  putStrLn $ "length:  " ++ show (length myTree)
  putStrLn $ "elem 4:  " ++ show (elem 4 myTree)
  putStrLn $ "elem 6:  " ++ show (elem 6 myTree)

demo_tree

Line 31: Use camelCase
Found:
demo_tree :: IO ()
Why not:
demoTree :: IO ()Line 32: Use camelCase
Found:
demo_tree = ...
Why not:
demoTree = ...Line 43: Use infix
Found:
elem 4 myTree
Why not:
4 `elem` myTreeLine 44: Use infix
Found:
elem 6 myTree
Why not:
6 `elem` myTree


=== BTree — три порядка обхода ===
in-order  (toList): [1,3,4,5,7,8,9]
pre-order:          [5,3,1,4,8,7,9]
post-order:         [1,4,3,7,9,8,5]

sum:     37
product: 30240
maximum: 9
minimum: 1
length:  7
elem 4:  True
elem 6:  False

## 💡 Ловушки

- **`foldl` без строгости** — утечка памяти на больших списках. Всегда используй `foldl'` для числовых агрегаций.
- **`foldr` + `foldl`** возвращают разный результат для нечестных операций: `foldr (-) 0 [1,2,3] = 2`, `foldl (-) 0 [1,2,3] = -6`.
- **`maximum`/`minimum` на пустой структуре** бросают исключение `Prelude.maximum: empty list`. Используй `maximumMay` из библиотеки `safe` или ручную проверку.
- **Порядок обхода** зависит от реализации `foldr` — нет гарантий порядка для произвольных `Foldable`.
- **`length` через `Foldable`** имеет сложность O(n) для связных списков, но O(1) для `Data.Sequence`.

## 🔭 Категорная точка зрения

`Foldable t` — это натуральное преобразование `t a → List a` (функция `toList`).  
Более формально: это F-алгебра, где `F` — это сигнатура списка `[a] = 1 + a × [a]`.

```
foldMap :: Monoid m => (a -> m) -> t a -> m
        -- это единственный гомоморфизм из (t a, ?) в (m, <>)
        -- при условии что a -> m задаёт образующие
```

Связь с монадой:
```haskell
-- Любой Foldable + Monad дают эти операции:
mapM_  :: (Foldable t, Monad m) => (a -> m b) -> t a -> m ()
sequence_ :: (Foldable t, Monad m) => t (m a) -> m ()
```

---
# 2️⃣ `foldMap` и Моноиды — Сердце `Foldable`

## 📐 Теория

`foldMap` — это **самый мощный** метод `Foldable`. Все остальные выражаются через него:

```haskell
foldMap :: (Foldable t, Monoid m) => (a -> m) -> t a -> m
foldMap f = foldr ((<>) . f) mempty
```

**Идея:** преобразуй каждый элемент в моноид (`a -> m`), потом «склей» всё через `(<>)`.

Примеры моноидов для свёртки:

| Моноид | `mempty` | `(<>)` | Что считает |
|--------|---------|--------|-------------|
| `Sum Int` | `0` | `+` | Сумма |
| `Product Int` | `1` | `*` | Произведение |
| `[a]` | `[]` | `++` | Конкатенация |
| `Any` | `False` | `||` | Есть ли True |
| `All` | `True` | `&&` | Все ли True |
| `First a` | `Nothing` | взять первый | Первый Just |
| `Last a` | `Nothing` | взять последний | Последний Just |
| `Max a` | `-∞` | `max` | Максимум |
| `Min a` | `+∞` | `min` | Минимум |
| `Map k v` | `{}` | `unionWith` | Гистограмма |

## ⭐ Минимальная реализация через `foldMap`

```haskell
-- Из одного foldMap вытекает всё:
sum     t = getSum     $ foldMap Sum     t
product t = getProduct $ foldMap Product t
any  p  t = getAny     $ foldMap (Any . p) t
all  p  t = getAll     $ foldMap (All . p) t
toList  t = foldMap (:[]) t
length  t = getSum $ foldMap (\_ -> Sum 1) t
```

In [5]:
import Data.Monoid (Sum(..), Product(..), Any(..), All(..), First(..), Last(..))

-- Пример 1: foldMap с разными моноидами
demo_foldmap :: IO ()
demo_foldmap = do
  putHeader "foldMap с разными моноидами"
  let xs = [3, 1, 4, 1, 5, 9, 2, 6] :: [Int]

  putStrLn $ "Список: " ++ show xs
  putStrLn $ "Sum:     " ++ show (getSum     $ foldMap Sum     xs)
  putStrLn $ "Product: " ++ show (getProduct $ foldMap Product xs)
  putStrLn $ "Any>5:   " ++ show (getAny     $ foldMap (Any . (>5)) xs)
  putStrLn $ "All>0:   " ++ show (getAll     $ foldMap (All . (>0)) xs)
  putStrLn $ "toList:  " ++ show (foldMap (:[]) xs)
  putStrLn $ "length:  " ++ show (getSum $ foldMap (\_ -> Sum 1) xs)
  putStrLn $ "First>4: " ++ show (getFirst $ foldMap (\x -> First $ if x > 4 then Just x else Nothing) xs)
  putStrLn $ "Last<3:  " ++ show (getLast  $ foldMap (\x -> Last  $ if x < 3 then Just x else Nothing) xs)

demo_foldmap

Line 4: Use camelCase
Found:
demo_foldmap :: IO ()
Why not:
demoFoldmap :: IO ()Line 5: Use camelCase
Found:
demo_foldmap = ...
Why not:
demoFoldmap = ...


=== foldMap с разными моноидами ===
Список: [3,1,4,1,5,9,2,6]
Sum:     31
Product: 6480
Any>5:   True
All>0:   True
toList:  [3,1,4,1,5,9,2,6]
length:  8
First>4: Just 5
Last<3:  Just 2

In [6]:
-- Пример 2: foldMap для построения Map (гистограмма)
demo_histogram :: IO ()
demo_histogram = do
  putHeader "foldMap → гистограмма"
  let text = "the quick brown fox jumps over the lazy dog"
  let words' = words text

  -- Частота слов через foldMap
  let freq = foldMap (\w -> Map.singleton w 1) words' :: Map String (Sum Int)
  putStrLn "Частота слов:"
  mapM_ (\(w,Sum n) -> putStrLn $ "  " ++ w ++ ": " ++ show n) (Map.toAscList freq)

  -- Частота букв
  let charFreq = foldMap (\c -> Map.singleton c (Sum 1)) (filter isAlpha text)
  putStrLn "\nТоп-5 букв:"
  let sorted = take 5 $ sortBy (comparing (Down . snd)) (Map.toList charFreq)
  mapM_ (\(c, Sum n) -> putStrLn $ "  '" ++ [c] ++ "': " ++ show n) sorted

  -- Индексирование: слово → позиции
  let indexed = foldMap (\(i,w) -> Map.singleton w [i]) (zip [0..] words')
  putStrLn "\nПозиции слова 'the':"
  print $ Map.lookup "the" indexed

demo_histogram

Line 2: Use camelCase
Found:
demo_histogram :: IO ()
Why not:
demoHistogram :: IO ()Line 3: Use camelCase
Found:
demo_histogram = ...
Why not:
demoHistogram = ...Line 9: Avoid lambda using `infix`
Found:
(\ w -> Map.singleton w 1)
Why not:
(`Map.singleton` 1)


=== foldMap → гистограмма ===
Частота слов:
  brown: 1
  dog: 1
  fox: 1
  jumps: 1
  lazy: 1
  over: 1
  quick: 1
  the: 1

Топ-5 букв:
  'a': 1
  'b': 1
  'c': 1
  'd': 1
  'e': 1

Позиции слова 'the':
Just [0]

In [7]:
-- Пример 3: Свой Foldable через foldMap (розовое дерево)
data Rose a = RoseNode a [Rose a]
  deriving (Show, Functor)

instance Foldable Rose where
  -- Через foldMap: корень + рекурсия по детям
  foldMap f (RoseNode x children) =
    f x <> foldMap (foldMap f) children

-- Пример файловой системы
type FileSize = Int
fileSystem :: Rose (String, FileSize)
fileSystem = RoseNode ("/", 0)
  [ RoseNode ("home", 0)
      [ RoseNode ("alice", 0)
          [ RoseNode ("docs", 0)
              [ RoseNode ("report.pdf", 1500) []
              , RoseNode ("notes.txt", 12) []
              ]
          , RoseNode ("photo.jpg", 3200) []
          ]
      , RoseNode ("bob", 0)
          [ RoseNode ("code.hs", 450) []
          ]
      ]
  , RoseNode ("usr", 0)
      [ RoseNode ("bin", 0)
          [ RoseNode ("bash", 800) []
          , RoseNode ("ls", 120) []
          ]
      ]
  ]

demo_rose :: IO ()
demo_rose = do
  putHeader "Rose Tree — файловая система"
  -- Все файлы (размер > 0)
  let files = filter (\(_,s) -> s > 0) (toList fileSystem)
  putStrLn "Все файлы:"
  mapM_ (\(n,s) -> putStrLn $ "  " ++ n ++ " (" ++ show s ++ " KB)") files
  putStrLn $ "\nОбщий размер: " ++ show (getSum $ foldMap (Sum . snd) fileSystem) ++ " KB"
  putStrLn $ "Кол-во файлов: " ++ show (length files)
  putStrLn $ "Самый большой: " ++ show (maximumBy (comparing snd) files)

demo_rose

Line 34: Use camelCase
Found:
demo_rose :: IO ()
Why not:
demoRose :: IO ()Line 35: Use camelCase
Found:
demo_rose = ...
Why not:
demoRose = ...


=== Rose Tree — файловая система ===
Все файлы:
  report.pdf (1500 KB)
  notes.txt (12 KB)
  photo.jpg (3200 KB)
  code.hs (450 KB)
  bash (800 KB)
  ls (120 KB)

Общий размер: 6082 KB
Кол-во файлов: 6
Самый большой: ("photo.jpg",3200)

## 💡 Ловушки `foldMap`

- **`foldMap` не строгий по умолчанию** — для числовых агрегаций используй `foldMap'` (из `Data.Foldable`, GHC ≥ 9.2).
- **`First`/`Last` из `Data.Monoid`** работают через `Maybe`, но их семантика `Last` — `(<>)` берёт *правый* операнд, если оба `Just`. Не путай с `Data.Semigroup.Last`.
- **Порядок `(<>)`** в `foldMap` определяется реализацией `Foldable` (обычно слева направо для списков).
- **`fold = foldMap id`** — полезно когда элементы уже моноиды.

---
# 3️⃣ Встроенные функции `Foldable`

## 📐 Полный арсенал

```haskell
-- Агрегация
sum, product     :: (Foldable t, Num a) => t a -> a
maximum, minimum :: (Foldable t, Ord a) => t a -> a
maximumBy, minimumBy :: Foldable t => (a->a->Ordering) -> t a -> a

-- Предикаты
any, all   :: Foldable t => (a -> Bool) -> t a -> Bool
elem       :: (Foldable t, Eq a) => a -> t a -> Bool
notElem    :: (Foldable t, Eq a) => a -> t a -> Bool

-- Структура
null       :: Foldable t => t a -> Bool
length     :: Foldable t => t a -> Int
toList     :: Foldable t => t a -> [a]

-- Поиск
find       :: Foldable t => (a -> Bool) -> t a -> Maybe a
asum       :: (Foldable t, Alternative f) => t (f a) -> f a

-- Действия с эффектами
for_       :: (Foldable t, Applicative f) => t a -> (a -> f b) -> f ()
traverse_  :: (Foldable t, Applicative f) => (a -> f b) -> t a -> f ()
mapM_      :: (Foldable t, Monad m) => (a -> m b) -> t a -> m ()
sequenceA_ :: (Foldable t, Applicative f) => t (f a) -> f ()
sequence_  :: (Foldable t, Monad m) => t (m a) -> m ()

-- Конкатенация
concat     :: Foldable t => t [a] -> [a]  -- = fold
concatMap  :: Foldable t => (a -> [b]) -> t a -> [b]  -- = foldMap
```

In [8]:
-- Пример 1: maximumBy/minimumBy — свои компараторы
data Person = Person { name :: String, age :: Int, score :: Double }
  deriving (Show)

people :: [Person]
people =
  [ Person "Alice" 30 9.5
  , Person "Bob"   25 7.2
  , Person "Carol" 35 8.8
  , Person "Dave"  28 9.1
  , Person "Eve"   22 6.5
  ]

-- find — первый элемент, удовлетворяющий предикату
findFirst :: Foldable t => (a -> Bool) -> t a -> Maybe a
findFirst p = getFirst . foldMap (\x -> First $ if p x then Just x else Nothing)

demo_maxmin :: IO ()
demo_maxmin = do
  putHeader "maximumBy / minimumBy"

  let oldest   = maximumBy (comparing age)   people
  let youngest = minimumBy (comparing age)   people
  let topScore = maximumBy (comparing score) people

  putStrLn $ "Старший:  " ++ name oldest  ++ " (" ++ show (age oldest) ++ ")"
  putStrLn $ "Младший:  " ++ name youngest ++ " (" ++ show (age youngest) ++ ")"
  putStrLn $ "Топ-скор: " ++ name topScore ++ " (" ++ show (score topScore) ++ ")"

  let sorted = sortBy (comparing (Down . score)) people
  putStrLn "\nРейтинг по score:"
  mapM_ (\p -> putStrLn $ "  " ++ name p ++ ": " ++ show (score p)) sorted

  let found = findFirst (\p -> score p > 9.0) people
  putStrLn $ "\nПервый с score > 9.0: " ++ maybe "не найден" name found

demo_maxmin

Line 18: Use camelCase
Found:
demo_maxmin :: IO ()
Why not:
demoMaxmin :: IO ()Line 19: Use camelCase
Found:
demo_maxmin = ...
Why not:
demoMaxmin = ...


=== maximumBy / minimumBy ===
Старший:  Carol (35)
Младший:  Eve (22)
Топ-скор: Alice (9.5)

Рейтинг по score:
  Alice: 9.5
  Dave: 9.1
  Carol: 8.8
  Bob: 7.2
  Eve: 6.5

Первый с score > 9.0: Alice

In [9]:
-- Пример 2: for_ / traverse_ — эффекты без сохранения результата
demo_effects :: IO ()
demo_effects = do
  putHeader "for_ / traverse_ — обход с эффектами"

  -- for_ = flip traverse_ = "для каждого делай IO"
  putStrLn "for_ по списку:"
  for_ [1..5 :: Int] $ \i -> do
    putStrLn $ "  Обрабатываем элемент " ++ show i

  -- traverse_ по Maybe
  putStrLn "\ntraverse_ по Just 42:"
  traverse_ print (Just (42 :: Int))

  putStrLn "traverse_ по Nothing:"
  traverse_ print (Nothing :: Maybe Int)
  putStrLn "  (ничего не выводит)"

  -- asum — первый успешный
  putHeader "asum — первый Just"
  let maybes = [Nothing, Nothing, Just 42, Just 99, Nothing] :: [Maybe Int]
  putStrLn $ "asum " ++ show maybes ++ " = " ++ show (asum maybes)

  -- sequence_ для IO-действий
  let actions = [putStr "H", putStr "e", putStr "l", putStr "l", putStr "o"]
  putStr "\nsequence_: "
  sequence_ actions
  putStrLn ""

demo_effects

Line 2: Use camelCase
Found:
demo_effects :: IO ()
Why not:
demoEffects :: IO ()Line 3: Use camelCase
Found:
demo_effects = ...
Why not:
demoEffects = ...


=== for_ / traverse_ — обход с эффектами ===
for_ по списку:
  Обрабатываем элемент 1
  Обрабатываем элемент 2
  Обрабатываем элемент 3
  Обрабатываем элемент 4
  Обрабатываем элемент 5

traverse_ по Just 42:
42
traverse_ по Nothing:
  (ничего не выводит)

=== asum — первый Just ===
asum [Nothing,Nothing,Just 42,Just 99,Nothing] = Just 42

sequence_: Hello

In [10]:
-- Пример 3: concat / concatMap для вложенных структур
demo_concat :: IO ()
demo_concat = do
  putHeader "concat / concatMap"

  -- concat
  let nested = [[1,2,3], [4,5], [6,7,8,9]] :: [[Int]]
  putStrLn $ "concat " ++ show nested ++ " = " ++ show (concat nested)

  -- concatMap = foldMap
  let expand n = [1..n]
  putStrLn $ "concatMap (\\n -> [1..n]) [1..5] = " ++ show (concatMap expand [1..5 :: Int])

  -- Моноидная свёртка строк
  let wordsList = [["hello", "world"], ["foo", "bar"], ["baz"]]
  putStrLn $ "\nfold [[\"hello\",\"world\"],...] = " ++ show (fold wordsList)

  -- Проверка на дерево
  let t = BNode (BNode BLeaf [1,2] BLeaf) [3] (BNode BLeaf [4,5] BLeaf)
  putStrLn $ "\nconcatMap дерева: " ++ show (concatMap id t)

  -- any / all
  putHeader "any / all"
  let nums = [2, 4, 6, 8, 10] :: [Int]
  putStrLn $ "any odd  " ++ show nums ++ " = " ++ show (any odd nums)
  putStrLn $ "all even " ++ show nums ++ " = " ++ show (all even nums)
  putStrLn $ "any (>9) " ++ show nums ++ " = " ++ show (any (>9) nums)
  putStrLn $ "all (>0) " ++ show nums ++ " = " ++ show (all (>0) nums)

demo_concat

Line 2: Use camelCase
Found:
demo_concat :: IO ()
Why not:
demoConcat :: IO ()Line 3: Use camelCase
Found:
demo_concat = ...
Why not:
demoConcat = ...Line 20: Use concat
Found:
concatMap id
Why not:
concat


=== concat / concatMap ===
concat [[1,2,3],[4,5],[6,7,8,9]] = [1,2,3,4,5,6,7,8,9]
concatMap (\n -> [1..n]) [1..5] = [1,1,2,1,2,3,1,2,3,4,1,2,3,4,5]

fold [["hello","world"],...] = ["hello","world","foo","bar","baz"]

concatMap дерева: [1,2,3,4,5]

=== any / all ===
any odd  [2,4,6,8,10] = False
all even [2,4,6,8,10] = True
any (>9) [2,4,6,8,10] = True
all (>0) [2,4,6,8,10] = True

---
# 4️⃣ `Foldable` для Своих Типов

## 📐 Когда нужно писать instance вручную

GHC может автоматически вывести `Foldable` через `{-# LANGUAGE DeriveFoldable #-}`, но:
- Автовывод обходит в **левый-правый порядок** слева направо по полям конструктора
- Иногда нужен нестандартный порядок (pre-order, BFS, reverse)
- Иногда структура нерегулярная (граф, матрица, очередь)

```haskell
-- Автовывод:
data Tree a = Leaf | Node (Tree a) a (Tree a)
  deriving (Foldable)  -- in-order: left, a, right

-- Вручную (можно изменить порядок):
instance Foldable Tree where
  foldMap f Leaf         = mempty
  foldMap f (Node l x r) = foldMap f l <> f x <> foldMap f r
```

## ⭐ Правило

Для `foldMap f t`: каждый элемент `a` в `t` должен появиться в результате **ровно один раз**.

In [11]:
-- Пример 1: BFS-обход через очередь
data BinTree a = BNil | BN (BinTree a) a (BinTree a) deriving Show

-- Functor instance
instance Functor BinTree where
  fmap _ BNil       = BNil
  fmap f (BN l x r) = BN (fmap f l) (f x) (fmap f r)

-- Foldable instance (in-order)
instance Foldable BinTree where
  foldMap _ BNil       = mempty
  foldMap f (BN l x r) = foldMap f l <> f x <> foldMap f r

-- Дополнительные функции обхода
foldMapInOrder :: Monoid m => (a -> m) -> BinTree a -> m
foldMapInOrder = foldMap

foldMapPreOrder :: Monoid m => (a -> m) -> BinTree a -> m
foldMapPreOrder _ BNil       = mempty
foldMapPreOrder f (BN l x r) = f x <> foldMapPreOrder f l <> foldMapPreOrder f r

-- BFS через уровни
bfsLevels :: BinTree a -> [[a]]
bfsLevels BNil = []
bfsLevels t = go [t]
  where
    go [] = []
    go level =
      let values   = [x | BN _ x _ <- level]
          children = concatMap getKids level
      in if null values then [] else values : go children
    getKids BNil       = []
    getKids (BN l _ r) = [l, r]

bfsOrder :: BinTree a -> [a]
bfsOrder = concat . bfsLevels

-- Утилита для построения дерева
buildBinTree :: Int -> BinTree Int
buildBinTree 0 = BNil
buildBinTree n = BN (buildBinTree (n-1)) n (buildBinTree (n-1))

-- Пример дерева: 1 / 2 5 / 4 5 3 6
sampleBT :: BinTree Int
sampleBT = BN (BN (BN BNil 4 BNil) 2 (BN BNil 5 BNil)) 1 (BN BNil 3 (BN BNil 6 BNil))

demo_traversal_orders :: IO ()
demo_traversal_orders = do
  putHeader "Порядки обхода дерева"
  putStrLn $ "in-order:  " ++ show (foldMapInOrder (:[]) sampleBT)
  putStrLn $ "pre-order: " ++ show (foldMapPreOrder (:[]) sampleBT)
  putStrLn $ "BFS:       " ++ show (bfsOrder sampleBT)
  putStrLn $ "BFS уровни: " ++ show (bfsLevels sampleBT)

demo_traversal_orders

Line 47: Use camelCase
Found:
demo_traversal_orders :: IO ()
Why not:
demoTraversalOrders :: IO ()Line 48: Use camelCase
Found:
demo_traversal_orders = ...
Why not:
demoTraversalOrders = ...


=== Порядки обхода дерева ===
in-order:  [4,2,5,1,3,6]
pre-order: [1,2,4,5,3,6]
BFS:       [1,2,3,4,5,6]
BFS уровни: [[1],[2,3],[4,5,6]]

In [12]:
-- Пример 2: Foldable для матрицы (2D-сетки)
newtype Matrix a = Matrix { unMatrix :: [[a]] } deriving (Show)

instance Functor Matrix where
  fmap f (Matrix rows) = Matrix (map (map f) rows)

instance Foldable Matrix where
  foldMap f (Matrix rows) = foldMap (foldMap f) rows

-- Вспомогательные функции
matrixSize :: Matrix a -> (Int, Int)
matrixSize (Matrix rows) = (length rows, if null rows then 0 else length (head rows))

showMatrix :: Show a => Matrix a -> String
showMatrix (Matrix rows) = unlines $ map (intercalate "\t" . map show) rows

demo_matrix :: IO ()
demo_matrix = do
  putHeader "Foldable для Matrix"
  let m = Matrix [[1,2,3],[4,5,6],[7,8,9]] :: Matrix Int
  putStrLn $ "Матрица " ++ show (matrixSize m) ++ ":"
  putStr (showMatrix m)
  putStrLn $ "sum:     " ++ show (sum m)
  putStrLn $ "maximum: " ++ show (maximum m)
  putStrLn $ "toList:  " ++ show (toList m)
  putStrLn $ "any (>7): " ++ show (any (>7) m)
  putStrLn $ "length:  " ++ show (length m)

  -- Главная диагональ через zipWith
  let diag (Matrix rows) = zipWith (!!) rows [0..]
  putStrLn $ "Диагональ: " ++ show (diag m)

demo_matrix

Line 17: Use camelCase
Found:
demo_matrix :: IO ()
Why not:
demoMatrix :: IO ()Line 18: Use camelCase
Found:
demo_matrix = ...
Why not:
demoMatrix = ...


=== Foldable для Matrix ===
Матрица (3,3):
1	2	3
4	5	6
7	8	9
sum:     45
maximum: 9
toList:  [1,2,3,4,5,6,7,8,9]
any (>7): True
length:  9
Диагональ: [1,5,9]

In [13]:
-- Пример 3: Foldable для пары (только правый элемент!)
-- (,) e уже имеет Foldable в Haskell: foldMap f (_, x) = f x

demo_pair_foldable :: IO ()
demo_pair_foldable = do
  putHeader "Foldable для пары (,)"
  -- Пара — это 'контейнер' для второго элемента
  let p = ("ignored", 42) :: (String, Int)
  putStrLn $ "toList (\"ignored\", 42) = " ++ show (toList p)   -- [42]
  putStrLn $ "sum    (\"ignored\", 42) = " ++ show (sum p)
  putStrLn $ "length (\"ignored\", 42) = " ++ show (length p)  -- 1

  -- Foldable для Either: Right содержит, Left пустой
  let r = Right 42 :: Either String Int
  let l = Left "oops" :: Either String Int
  putStrLn $ "\ntoList (Right 42) = " ++ show (toList r)  -- [42]
  putStrLn $ "toList (Left ..)  = " ++ show (toList l)   -- []
  putStrLn $ "sum    (Right 42) = " ++ show (sum r)      -- 42
  putStrLn $ "null   (Left ..)  = " ++ show (null l)     -- True

  -- Maybe: Just = один элемент, Nothing = пустой
  putStrLn $ "\nsum    (Just 5)  = " ++ show (sum (Just (5::Int)))
  putStrLn $ "sum    Nothing   = " ++ show (sum (Nothing :: Maybe Int))
  putStrLn $ "toList (Just 'a') = " ++ show (toList (Just 'a'))

demo_pair_foldable

Line 4: Use camelCase
Found:
demo_pair_foldable :: IO ()
Why not:
demoPairFoldable :: IO ()Line 5: Use camelCase
Found:
demo_pair_foldable = ...
Why not:
demoPairFoldable = ...


=== Foldable для пары (,) ===
toList ("ignored", 42) = [42]
sum    ("ignored", 42) = 42
length ("ignored", 42) = 1

toList (Right 42) = [42]
toList (Left ..)  = []
sum    (Right 42) = 42
null   (Left ..)  = True

sum    (Just 5)  = 5
sum    Nothing   = 0
toList (Just 'a') = "a"

---
# 5️⃣ `Traversable` — Основы

## 📐 Теория

`Traversable` расширяет `Foldable` и `Functor`: позволяет **проходить по структуре с эффектами**, сохраняя её форму.

```haskell
class (Functor t, Foldable t) => Traversable t where
  -- Минимальная реализация: traverse ИЛИ sequenceA
  traverse  :: Applicative f => (a -> f b) -> t a -> f (t b)
  sequenceA :: Applicative f => t (f a) -> f (t a)

  -- Монадические версии (обычно = traverse/sequenceA)
  mapM     :: Monad m => (a -> m b) -> t a -> m (t b)
  sequence :: Monad m => t (m a) -> m (t a)
```

**Интуиция:** `traverse` — это как `fmap`, но функция `a -> f b` возвращает **контейнер с эффектом**. `Traversable` «вытаскивает» эффект наружу, сохраняя структуру:

```
fmap     :: (a -> b)   -> t a -> t b      -- без эффектов
traverse :: (a -> f b) -> t a -> f (t b)  -- с эффектом f
```

**`sequenceA`** меняет местами два функтора:
```
sequenceA :: t (f a) -> f (t a)
           -- [Maybe a] -> Maybe [a]  (все Just или Nothing)
           -- [IO a]    -> IO [a]     (запустить все IO и собрать)
           -- [Either e a] -> Either e [a]
```

## ⭐ Законы

```haskell
-- 1. Тождество
traverse Identity = Identity             -- без эффекта = fmap

-- 2. Композиция
traverse (Compose . fmap g . f) = Compose . fmap (traverse g) . traverse f

-- 3. Натуральность
t . traverse f = traverse (t . f)   -- для натурального t
```

Из законов следует: `traverse` **не меняет форму** структуры (количество и расположение элементов).

In [14]:
-- Пример 1: sequenceA — "перестановка" функторов
demo_sequenceA :: IO ()
demo_sequenceA = do
  putHeader "sequenceA — меняем местами функторы"

  -- [Maybe a] -> Maybe [a]
  let allJust  = [Just 1, Just 2, Just 3] :: [Maybe Int]
  let hasNothing = [Just 1, Nothing, Just 3] :: [Maybe Int]
  putStrLn $ "sequenceA [Just 1, Just 2, Just 3]   = " ++ show (sequenceA allJust)
  putStrLn $ "sequenceA [Just 1, Nothing, Just 3]  = " ++ show (sequenceA hasNothing)

  -- [Either e a] -> Either e [a]
  let allRight = [Right 1, Right 2, Right 3] :: [Either String Int]
  let hasLeft  = [Right 1, Left "oops", Right 3] :: [Either String Int]
  putStrLn $ "\nsequenceA [Right 1, Right 2, Right 3]  = " ++ show (sequenceA allRight)
  putStrLn $ "sequenceA [Right 1, Left \"oops\", ...]  = " ++ show (sequenceA hasLeft)

  -- [[a]] -> [[a]] — декартово произведение!
  let lists = [[1,2],[3,4],[5,6]] :: [[Int]]
  putStrLn $ "\nsequenceA [[1,2],[3,4],[5,6]] = декартово произведение:"
  mapM_ print (sequenceA lists)

demo_sequenceA

Line 2: Use camelCase
Found:
demo_sequenceA :: IO ()
Why not:
demoSequenceA :: IO ()Line 3: Use camelCase
Found:
demo_sequenceA = ...
Why not:
demoSequenceA = ...Line 20: Redundant $
Found:
putStrLn
  $ "\nsequenceA [[1,2],[3,4],[5,6]] = декартово произведение:"
Why not:
putStrLn
  "\nsequenceA [[1,2],[3,4],[5,6]] = декартово произведение:"


=== sequenceA — меняем местами функторы ===
sequenceA [Just 1, Just 2, Just 3]   = Just [1,2,3]
sequenceA [Just 1, Nothing, Just 3]  = Nothing

sequenceA [Right 1, Right 2, Right 3]  = Right [1,2,3]
sequenceA [Right 1, Left "oops", ...]  = Left "oops"

sequenceA [[1,2],[3,4],[5,6]] = декартово произведение:
[1,3,5]
[1,3,6]
[1,4,5]
[1,4,6]
[2,3,5]
[2,3,6]
[2,4,5]
[2,4,6]

In [15]:
-- Пример 2: traverse с разными эффектами
import Text.Read (readMaybe)

demo_traverse :: IO ()
demo_traverse = do
  putHeader "traverse с разными эффектами"

  -- traverse с Maybe — валидация
  let parseNum :: String -> Maybe Int
      parseNum = readMaybe

  let goodStrings = ["1", "2", "3", "4", "5"]
  let badStrings  = ["1", "2", "abc", "4"]
  putStrLn $ "traverse readMaybe [\"1\"..\"5\"] = " ++ show (traverse parseNum goodStrings)
  putStrLn $ "traverse readMaybe [\"1\",\"abc\",...] = " ++ show (traverse parseNum badStrings)

  -- traverse с Either — валидация с ошибкой
  let parsePositive :: Int -> Either String Int
      parsePositive n
        | n > 0    = Right n
        | otherwise = Left $ show n ++ " не является положительным"

  putStrLn $ "\ntraverse parsePositive [1,2,3] = " ++ show (traverse parsePositive [1,2,3])
  putStrLn $ "traverse parsePositive [1,-2,3] = " ++ show (traverse parsePositive [1,-2,3])

  -- traverse с IO
  putHeader "traverse с IO"
  results <- traverse (\n -> do
    let sq = n * n :: Int
    putStrLn $ "  Вычисляю " ++ show n ++ "^2 = " ++ show sq
    return sq) [1..5]
  putStrLn $ "Результаты: " ++ show results

demo_traverse

Line 4: Use camelCase
Found:
demo_traverse :: IO ()
Why not:
demoTraverse :: IO ()Line 5: Use camelCase
Found:
demo_traverse = ...
Why not:
demoTraverse = ...


=== traverse с разными эффектами ===
traverse readMaybe ["1".."5"] = Just [1,2,3,4,5]
traverse readMaybe ["1","abc",...] = Nothing

traverse parsePositive [1,2,3] = Right [1,2,3]
traverse parsePositive [1,-2,3] = Left "-2 \1085\1077 \1103\1074\1083\1103\1077\1090\1089\1103 \1087\1086\1083\1086\1078\1080\1090\1077\1083\1100\1085\1099\1084"

=== traverse с IO ===
  Вычисляю 1^2 = 1
  Вычисляю 2^2 = 4
  Вычисляю 3^2 = 9
  Вычисляю 4^2 = 16
  Вычисляю 5^2 = 25
Результаты: [1,4,9,16,25]

In [16]:
-- Пример 3: traverse = fmap + sequenceA
-- Закон Identity: traverse Identity = fmap

newtype MyIdentity a = MyIdentity { runMyIdentity :: a }

instance Functor MyIdentity where fmap f (MyIdentity x) = MyIdentity (f x)
instance Applicative MyIdentity where
  pure = MyIdentity
  MyIdentity f <*> MyIdentity x = MyIdentity (f x)

demo_traverse_decompose :: IO ()
demo_traverse_decompose = do
  putHeader "traverse = sequenceA . fmap f"

  let xs = [1, 2, 3, 4, 5] :: [Int]
  let f x = if even x then Just (x * 10) else Just (x * 100)

  let r1 = traverse f xs
  let r2 = sequenceA (fmap f xs)

  putStrLn $ "traverse f [1..5]            = " ++ show r1
  putStrLn $ "sequenceA . fmap f $ [1..5]  = " ++ show r2
  putStrLn $ "Равны: " ++ show (r1 == r2)

  putHeader "Закон Identity"
  let idResult  = runMyIdentity (traverse (MyIdentity . (*2)) ([1..5] :: [Int]))
  let fmapResult = fmap (*2) ([1..5] :: [Int])
  putStrLn $ "traverse Identity (*2) [1..5] = " ++ show idResult
  putStrLn $ "fmap (*2) [1..5]               = " ++ show fmapResult
  putStrLn $ "Равны: " ++ show (idResult == fmapResult)

demo_traverse_decompose

Line 11: Use camelCase
Found:
demo_traverse_decompose :: IO ()
Why not:
demoTraverseDecompose :: IO ()Line 12: Use camelCase
Found:
demo_traverse_decompose = ...
Why not:
demoTraverseDecompose = ...Line 19: Use traverse
Found:
sequenceA (fmap f xs)
Why not:
traverse f xs


=== traverse = sequenceA . fmap f ===
traverse f [1..5]            = Just [100,20,300,40,500]
sequenceA . fmap f $ [1..5]  = Just [100,20,300,40,500]
Равны: True

=== Закон Identity ===
traverse Identity (*2) [1..5] = [2,4,6,8,10]
fmap (*2) [1..5]               = [2,4,6,8,10]
Равны: True

## Traversable и Коммутирование Функторов

`sequenceA :: (Traversable t, Applicative f) => t (f a) -> f (t a)`

Коммутатор двух функторов: `t (f a) --> f (t a)`

**Законы:**
- `traverse f = sequenceA . fmap f`
- `sequenceA = traverse id`

(McBride & Paterson, 2008): Traversable = функтор, коммутирующий с любым Applicative.

In [17]:
-- sequenceA
demo_commute :: IO ()
demo_commute = do
  putHeader "sequenceA: коммутатор функторов"
  let ms1 = [Just 1, Just 2, Just 3] :: [Maybe Int]
  let ms2 = [Just 1, Nothing, Just 3] :: [Maybe Int]
  putStrLn $ "sequenceA [J1,J2,J3] = " ++ show (sequenceA ms1)
  putStrLn $ "sequenceA [J1,N,J3] = " ++ show (sequenceA ms2)
  let xss = [[1,2],[3,4]] :: [[Int]]
  putStrLn $ "sequenceA [[1,2],[3,4]] = " ++ show (sequenceA xss)
  putStrLn $ "traverse id == sequenceA: " ++ show (traverse id ms1 == sequenceA ms1)
  let f x = if x > 0 then Just x else Nothing
  let xs = [1, 2, -3, 4] :: [Int]
  putStrLn $ "traverse f xs = " ++ show (traverse f xs)
  putStrLn $ "(seqA.fmap f) xs = " ++ show ((sequenceA . fmap f) xs)
  putStrLn $ "Equal: " ++ show (traverse f xs == (sequenceA . fmap f) xs)
demo_commute


Line 2: Use camelCase
Found:
demo_commute :: IO ()
Why not:
demoCommute :: IO ()Line 3: Use camelCase
Found:
demo_commute = ...
Why not:
demoCommute = ...Line 11: Use sequenceA
Found:
traverse id
Why not:
sequenceALine 15: Use traverse
Found:
sequenceA . fmap f
Why not:
traverse fLine 16: Use traverse
Found:
sequenceA . fmap f
Why not:
traverse f


=== sequenceA: коммутатор функторов ===
sequenceA [J1,J2,J3] = Just [1,2,3]
sequenceA [J1,N,J3] = Nothing
sequenceA [[1,2],[3,4]] = [[1,3],[1,4],[2,3],[2,4]]
traverse id == sequenceA: True
traverse f xs = Nothing
(seqA.fmap f) xs = Nothing
Equal: True

## Witherable

Witherable = Traversable + фильтрация за один проход

```haskell
wither :: Applicative f => (a -> f (Maybe b)) -> t a -> f (t b)
-- по ум.: fmap catMaybes . traverse f
```

| Операция | Тип |
|---|---|
| traverse | (a->fb)->ta->f(tb) |
| filter | (a->Bool)->ta->ta |
| wither | (a->f(Maybe b))->ta->f(tb) |

wither делает трансформацию + фильтрацию за один цикл.

In [18]:
import Data.Maybe (catMaybes)

wither :: Applicative f => (a -> f (Maybe b)) -> [a] -> f [b]
wither f xs = fmap catMaybes (traverse f xs)

validateDouble :: Int -> Maybe (Maybe Int)
validateDouble x
  | x > 0     = Just (Just (x*2))
  | otherwise = Just Nothing

demo_wither :: IO ()
demo_wither = do
  putHeader "Witherable: трансформ + фильтр за один проход"
  let xs = [-1, 2, -3, 4, 5] :: [Int]
  putStrLn $ "Вход: " ++ show xs
  putStrLn $ "wither validateDouble: " ++ show (wither validateDouble xs)
  result2 <- wither (\x -> if x>0 then return (Just x) else return Nothing) xs
  putStrLn $ "IO wither: " ++ show result2
demo_wither


Line 11: Use camelCase
Found:
demo_wither :: IO ()
Why not:
demoWither :: IO ()Line 12: Use camelCase
Found:
demo_wither = ...
Why not:
demoWither = ...


=== Witherable: трансформ + фильтр за один проход ===
Вход: [-1,2,-3,4,5]
wither validateDouble: Just [4,8,10]
IO wither: [2,4,5]

## 💡 Ловушки `Traversable`

- **`traverse` с `[]`** (списком как аппликативом) даёт декартово произведение — часто неожиданно: `traverse (\x -> [x, -x]) [1,2]` = `[[1,2],[1,-2],[-1,2],[-1,-2]]`.
- **`sequenceA` на `[Maybe a]`**: если хотя бы один `Nothing` — результат `Nothing`. Это «fail-fast» семантика.
- **`Either` тоже fail-fast**: `traverse` возвращает первую ошибку, не все. Для сбора всех ошибок нужна `Validation` / `These`.
- **`mapM` ≠ `traverse`** только теоретически: в GHC они синонимы через `Monad`, но `traverse` мощнее — работает с любым `Applicative`, не только `Monad`.

---
# 6️⃣ `Traversable` для Своих Типов

## 📐 Как реализовать `traverse`

```haskell
-- Паттерн: применяем f к каждому элементу, собираем через <*>
instance Traversable Tree where
  traverse _ Leaf         = pure Leaf
  traverse f (Node l x r) =
    Node <$> traverse f l <*> f x <*> traverse f r
```

**Ключевые моменты:**
- `pure Leaf` — «пустой» результат с тем же эффектом
- `<$>` и `<*>` — аппликативный стиль для «сборки» результата
- Порядок `<*>` определяет порядок выполнения эффектов

**Закон структуры:** `fmap f = runIdentity . traverse (Identity . f)`

In [19]:
-- Traversable для бинарного дерева
data TTree a = TLeaf | TNode (TTree a) a (TTree a)
  deriving (Show, Functor, Foldable)

instance Traversable TTree where
  traverse _ TLeaf         = pure TLeaf
  traverse f (TNode l x r) =
    TNode <$> traverse f l <*> f x <*> traverse f r

-- Пример дерева
numTree :: TTree Int
numTree = TNode
  (TNode (TNode TLeaf 1 TLeaf) 3 (TNode TLeaf 4 TLeaf))
  5
  (TNode (TNode TLeaf 7 TLeaf) 8 (TNode TLeaf 9 TLeaf))

demo_tree_traverse :: IO ()
demo_tree_traverse = do
  putHeader "Traversable TTree"

  -- traverse с Maybe: все узлы д.б. чётными?
  let evenOnly n = if even n then Just n else Nothing
  putStrLn $ "traverse evenOnly numTree = " ++ show (traverse evenOnly numTree)

  -- Всё чётное дерево
  let evenTree = TNode (TNode TLeaf 2 TLeaf) 4 (TNode TLeaf 6 TLeaf)
  putStrLn $ "traverse evenOnly evenTree = " ++ show (traverse evenOnly evenTree)

  -- traverse с IO — читаем метки
  putHeader "traverse с IO (нумерация узлов)"
  counter <- newIORef (0 :: Int)
  numbered <- traverse (\x -> do
    n <- readIORef counter
    writeIORef counter (n+1)
    return (n, x)) numTree
  putStrLn $ "Пронумерованное (in-order):"
  mapM_ (\(i,x) -> putStrLn $ "  #" ++ show i ++ " -> " ++ show x) (toList numbered)

demo_tree_traverse

Line 17: Use camelCase
Found:
demo_tree_traverse :: IO ()
Why not:
demoTreeTraverse :: IO ()Line 18: Use camelCase
Found:
demo_tree_traverse = ...
Why not:
demoTreeTraverse = ...Line 36: Redundant $
Found:
putStrLn $ "Пронумерованное (in-order):"
Why not:
putStrLn "Пронумерованное (in-order):"


=== Traversable TTree ===
traverse evenOnly numTree = Nothing
traverse evenOnly evenTree = Just (TNode (TNode TLeaf 2 TLeaf) 4 (TNode TLeaf 6 TLeaf))

=== traverse с IO (нумерация узлов) ===
Пронумерованное (in-order):
  #0 -> 1
  #1 -> 3
  #2 -> 4
  #3 -> 5
  #4 -> 7
  #5 -> 8
  #6 -> 9

In [20]:
-- Traversable для розового дерева
data RTree a = RNode a [RTree a]
  deriving (Show, Functor, Foldable)

instance Traversable RTree where
  traverse f (RNode x children) =
    RNode <$> f x <*> traverse (traverse f) children

-- Пример: валидация конфига
type Config = RTree (String, String)  -- (ключ, значение)

validateValue :: (String, String) -> Either String (String, Int)
validateValue (key, val) = case readMaybe val of
  Just n | n >= 0 -> Right (key, n)
  Just _          -> Left $ key ++ ": отрицательное значение '" ++ val ++ "'"
  Nothing         -> Left $ key ++ ": не число '" ++ val ++ "'"

goodConfig :: Config
goodConfig = RNode ("root", "0")
  [ RNode ("timeout", "30") []
  , RNode ("maxRetries", "3")
      [ RNode ("backoff", "5") []
      ]
  , RNode ("poolSize", "10") []
  ]

badConfig :: Config
badConfig = RNode ("root", "0")
  [ RNode ("timeout", "abc") []
  , RNode ("maxRetries", "-1") []
  ]

demo_rose_traverse :: IO ()
demo_rose_traverse = do
  putHeader "Traversable RTree — валидация конфига"
  putStrLn $ "Хороший конфиг: " ++ show (traverse validateValue goodConfig)
  putStrLn $ "Плохой конфиг:  " ++ show (traverse validateValue badConfig)

demo_rose_traverse

Line 33: Use camelCase
Found:
demo_rose_traverse :: IO ()
Why not:
demoRoseTraverse :: IO ()Line 34: Use camelCase
Found:
demo_rose_traverse = ...
Why not:
demoRoseTraverse = ...


=== Traversable RTree — валидация конфига ===
Хороший конфиг: Right (RNode ("root",0) [RNode ("timeout",30) [],RNode ("maxRetries",3) [RNode ("backoff",5) []],RNode ("poolSize",10) []])
Плохой конфиг:  Left "timeout: \1085\1077 \1095\1080\1089\1083\1086 'abc'"

In [21]:
-- Пример 3: Traversable для Maybe, Either, пары
demo_builtin_traversable :: IO ()
demo_builtin_traversable = do
  putHeader "traverse для встроенных типов"

  -- Maybe
  let safeRecip :: Double -> Maybe Double
      safeRecip 0 = Nothing
      safeRecip x = Just (1/x)

  putStrLn $ "traverse safeRecip (Just 4.0)  = " ++ show (traverse safeRecip (Just 4.0))
  putStrLn $ "traverse safeRecip (Just 0.0)  = " ++ show (traverse safeRecip (Just 0.0))
  putStrLn $ "traverse safeRecip Nothing      = " ++ show (traverse safeRecip (Nothing :: Maybe Double))

  -- Either
  let double n = Right (n * 2) :: Either String Int
  putStrLn $ "\ntraverse double (Right 5)  = " ++ show (traverse double (Right 5 :: Either String Int))
  putStrLn $ "traverse double (Left err) = " ++ show (traverse double (Left "err" :: Either String Int))

  -- (,) e — 'traverse только правого'
  let f x = [x, x*10] :: [Int]
  putStrLn $ "\ntraverse (\\x -> [x, x*10]) (\"tag\", 5) = " ++ show (traverse f ("tag", 5 :: Int))

  -- Пронумеровать элементы списка через State
  putHeader "Нумерация через State"
  let numberWith :: [a] -> [(Int, a)]
      numberWith xs = evalState (traverse step xs) 0
        where step x = do { n <- get; put (n+1); return (n, x) }
  putStrLn $ show $ numberWith ['a'..'e']

demo_builtin_traversable

Line 2: Use camelCase
Found:
demo_builtin_traversable :: IO ()
Why not:
demoBuiltinTraversable :: IO ()Line 3: Use camelCase
Found:
demo_builtin_traversable = ...
Why not:
demoBuiltinTraversable = ...Line 29: Use print
Found:
putStrLn $ show $ numberWith ['a' .. 'e']
Why not:
print (numberWith ['a' .. 'e'])


=== traverse для встроенных типов ===
traverse safeRecip (Just 4.0)  = Just (Just 0.25)
traverse safeRecip (Just 0.0)  = Nothing
traverse safeRecip Nothing      = Just Nothing

traverse double (Right 5)  = Right (Right 10)
traverse double (Left err) = Right (Left "err")

traverse (\x -> [x, x*10]) ("tag", 5) = [("tag",5),("tag",50)]

=== Нумерация через State ===
[(0,'a'),(1,'b'),(2,'c'),(3,'d'),(4,'e')]

---
# 7️⃣ Аппликативные Эффекты в `traverse`

## 📐 Теория

`traverse` работает с **любым** `Applicative` — это его суперсила:

| Аппликатив `f` | Эффект | Что делает `traverse f` |
|---------------|--------|------------------------|
| `IO` | Ввод-вывод | Выполняет IO для каждого элемента, собирает результаты |
| `Maybe` | Возможная ошибка | Возвращает `Nothing` при первом `Nothing` |
| `Either e` | Ошибка с сообщением | Возвращает `Left` при первой ошибке |
| `[]` | Недетерминизм | Декартово произведение |
| `State s` | Изменяемое состояние | Проходит с аккумулятором |
| `Writer w` | Логирование | Собирает лог по пути |
| `ZipList` | Параллельное применение | Попарно |

## ⭐ Ключевое свойство

Аппликатив `f` определяет **семантику** обхода. Одна и та же структура, разные эффекты — разные результаты.

In [22]:
-- Пример 1: traverse с State — нумерация, маркировка
demo_state_traverse :: IO ()
demo_state_traverse = do
  putHeader "traverse + State"

  let numberElems :: Traversable t => t a -> t (Int, a)
      numberElems t = evalState (traverse step t) 0
        where step x = do { n <- get; put (n+1); return (n, x) }

  putStrLn "Нумерованный список:"
  print $ numberElems ["alice", "bob", "carol"]

  putStrLn "Нумерованное дерево (in-order):"
  let t = TNode (TNode TLeaf 'a' TLeaf) 'b' (TNode TLeaf 'c' TLeaf)
  print $ numberElems t

  -- Замена нечётных с накоплением
  let replaceOdd :: [Int] -> ([Int], [Int])
      replaceOdd xs = runState (traverse step xs) []
        where step x
                | odd x = do
                    odds <- get
                    put (x:odds)
                    return 0
                | otherwise = return x

  let (result, odds) = replaceOdd [1..8]
  putStrLn $ "\nЗамена нечётных на 0: " ++ show result
  putStrLn $ "Нечётные:             " ++ show (reverse odds)

  putStrLn "\nДерево с DFS-индексами:"
  print $ numberElems numTree

demo_state_traverse

Line 2: Use camelCase
Found:
demo_state_traverse :: IO ()
Why not:
demoStateTraverse :: IO ()Line 3: Use camelCase
Found:
demo_state_traverse = ...
Why not:
demoStateTraverse = ...


=== traverse + State ===
Нумерованный список:
[(0,"alice"),(1,"bob"),(2,"carol")]
Нумерованное дерево (in-order):
TNode (TNode TLeaf (0,'a') TLeaf) (1,'b') (TNode TLeaf (2,'c') TLeaf)

Замена нечётных на 0: [0,2,0,4,0,6,0,8]
Нечётные:             [1,3,5,7]

Дерево с DFS-индексами:
TNode (TNode (TNode TLeaf (0,1) TLeaf) (1,3) (TNode TLeaf (2,4) TLeaf)) (3,5) (TNode (TNode TLeaf (4,7) TLeaf) (5,8) (TNode TLeaf (6,9) TLeaf))

In [23]:
-- Пример 2: traverse с Writer — логирование трансформаций
demo_writer_traverse :: IO ()
demo_writer_traverse = do
  putHeader "traverse + Writer (логирование)"

  let processNum :: Int -> Writer [String] Int
      processNum n
        | n < 0    = do { tell ["Отрицательный: " ++ show n ++ " → 0"]; return 0 }
        | n > 100  = do { tell ["Слишком большой: " ++ show n ++ " → 100"]; return 100 }
        | even n   = do { tell ["Чётный: " ++ show n ++ " → " ++ show (n*2)]; return (n*2) }
        | otherwise = return n

  let input = [-5, 3, 200, 4, 7, -1, 50] :: [Int]
  let (result, log) = runWriter $ traverse processNum input
  putStrLn $ "Вход:       " ++ show input
  putStrLn $ "Результат:  " ++ show result
  putStrLn   "Лог:"
  mapM_ (putStrLn . ("  " ++)) log

  -- Writer на дереве
  putHeader "traverse + Writer на дереве"
  let (processedTree, treeLog) = runWriter $ traverse processNum numTree
  putStrLn $ "Дерево: " ++ show (toList numTree) ++ " → " ++ show (toList processedTree)
  putStrLn $ "Лог:" ; mapM_ (putStrLn . ("  "++)) treeLog

demo_writer_traverse

Line 2: Use camelCase
Found:
demo_writer_traverse :: IO ()
Why not:
demoWriterTraverse :: IO ()Line 3: Use camelCase
Found:
demo_writer_traverse = ...
Why not:
demoWriterTraverse = ...Line 24: Redundant $
Found:
putStrLn $ "Лог:"
Why not:
putStrLn "Лог:"


=== traverse + Writer (логирование) ===
Вход:       [-5,3,200,4,7,-1,50]
Результат:  [0,3,100,8,7,0,100]
Лог:
  Отрицательный: -5 → 0
  Слишком большой: 200 → 100
  Чётный: 4 → 8
  Отрицательный: -1 → 0
  Чётный: 50 → 100

=== traverse + Writer на дереве ===
Дерево: [1,3,4,5,7,8,9] → [1,3,8,5,7,16,9]
Лог:
  Чётный: 4 → 8
  Чётный: 8 → 16

In [24]:
-- Пример 3: traverse с [] — недетерминизм
demo_list_traverse :: IO ()
demo_list_traverse = do
  putHeader "traverse с [] — недетерминизм"

  let expand :: Int -> [Int]
      expand n = [n, n*10, n*100]

  let r = traverse expand [1, 2] :: [[Int]]
  putStrLn "traverse (\\n -> [n, n*10, n*100]) [1, 2]:"
  putStrLn "-- Декартово произведение:"
  mapM_ print r

  let subsets xs = traverse (\x -> [[], [x]]) xs
  putStrLn $ "\nВсе подмножества [1,2,3]:"
  print $ map concat (subsets [1,2,3 :: Int])

  let perms [] = [[]]
      perms xs = [x:rest | x <- xs, rest <- perms (filter (/=x) xs)]
  putStrLn $ "\nВсе перестановки [1,2,3]:"
  print $ perms [1,2,3 :: Int]

demo_list_traverse

Line 2: Use camelCase
Found:
demo_list_traverse :: IO ()
Why not:
demoListTraverse :: IO ()Line 3: Use camelCase
Found:
demo_list_traverse = ...
Why not:
demoListTraverse = ...Line 14: Eta reduce
Found:
subsets xs = traverse (\ x -> [[], [x]]) xs
Why not:
subsets = traverse (\ x -> [[], [x]])Line 15: Redundant $
Found:
putStrLn $ "\nВсе подмножества [1,2,3]:"
Why not:
putStrLn "\nВсе подмножества [1,2,3]:"Line 20: Redundant $
Found:
putStrLn $ "\nВсе перестановки [1,2,3]:"
Why not:
putStrLn "\nВсе перестановки [1,2,3]:"


=== traverse с [] — недетерминизм ===
traverse (\n -> [n, n*10, n*100]) [1, 2]:
-- Декартово произведение:
[1,2]
[1,20]
[1,200]
[10,2]
[10,20]
[10,200]
[100,2]
[100,20]
[100,200]

Все подмножества [1,2,3]:
[[],[3],[2],[2,3],[1],[1,3],[1,2],[1,2,3]]

Все перестановки [1,2,3]:
[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]

---
# 8️⃣ `mapAccumL` / `mapAccumR` — Аккумулятор с Сохранением Формы

## 📐 Теория

```haskell
mapAccumL :: Traversable t
          => (s -> a -> (s, b))  -- функция: (аккумулятор, вход) -> (новый acc, выход)
          -> s                   -- начальный аккумулятор
          -> t a                 -- входная структура
          -> (s, t b)            -- (финальный acc, результат)

mapAccumR :: Traversable t
          => (s -> a -> (s, b))
          -> s
          -> t a
          -> (s, t b)            -- справа налево
```

**Интуиция:**
- `mapAccumL` = `traverse` с `State`, слева направо
- `mapAccumR` = `traverse` с `State`, справа налево
- Форма структуры сохраняется, аккумулятор течёт через неё

```
mapAccumL f z [a, b, c]
  z0 = z
  (z1, b1) = f z0 a
  (z2, b2) = f z1 b
  (z3, b3) = f z2 c
  результат = (z3, [b1, b2, b3])
```

In [25]:
-- Пример 1: mapAccumL — бегущий итог, нумерация
demo_mapaccum :: IO ()
demo_mapaccum = do
  putHeader "mapAccumL — левый аккумулятор"

  -- Бегущий итог
  let xs = [1, 2, 3, 4, 5] :: [Int]
  let (total, runningSum) = mapAccumL (\acc x -> let s = acc+x in (s, s)) 0 xs
  putStrLn $ "Список: " ++ show xs
  putStrLn $ "Бегущий итог: " ++ show runningSum
  putStrLn $ "Финальный: " ++ show total

  -- Нумерация с нуля
  let (n, numbered) = mapAccumL (\i x -> (i+1, (i, x))) 0 ['a'..'e']
  putStrLn $ "\nНумерация 'a'..'e': " ++ show numbered
  putStrLn $ "Количество: " ++ show n

  -- Замена на предыдущий элемент
  let (_, shifted) = mapAccumL (\prev x -> (x, prev)) 0 ([1..5] :: [Int])
  putStrLn $ "\nСдвиг вправо [1..5]: " ++ show shifted

  -- mapAccumR — справа налево
  putHeader "mapAccumR — правый аккумулятор"
  let (_, rShifted) = mapAccumR (\next x -> (x, next)) 0 ([1..5] :: [Int])
  putStrLn $ "Сдвиг влево [1..5]: " ++ show rShifted

  -- Суффиксные суммы
  let (_, sufSums) = mapAccumR (\acc x -> let s = acc+x in (s, s)) 0 ([1..5] :: [Int])
  putStrLn $ "Суффиксные суммы [1..5]: " ++ show sufSums

demo_mapaccum

Line 2: Use camelCase
Found:
demo_mapaccum :: IO ()
Why not:
demoMapaccum :: IO ()Line 3: Use camelCase
Found:
demo_mapaccum = ...
Why not:
demoMapaccum = ...


=== mapAccumL — левый аккумулятор ===
Список: [1,2,3,4,5]
Бегущий итог: [1,3,6,10,15]
Финальный: 15

Нумерация 'a'..'e': [(0,'a'),(1,'b'),(2,'c'),(3,'d'),(4,'e')]
Количество: 5

Сдвиг вправо [1..5]: [0,1,2,3,4]

=== mapAccumR — правый аккумулятор ===
Сдвиг влево [1..5]: [2,3,4,5,0]
Суффиксные суммы [1..5]: [15,14,12,9,5]

In [26]:
-- Пример 2: mapAccumL для сложных задач
demo_mapaccum2 :: IO ()
demo_mapaccum2 = do
  putHeader "mapAccumL — практические задачи"

  -- Уникальные метки (Map a Int)
  let labelUnique :: Ord a => [a] -> [Int]
      labelUnique xs = snd $ mapAccumL step Map.empty xs
        where
          step m x = case Map.lookup x m of
            Just i  -> (m, i)
            Nothing -> let i = Map.size m
                       in (Map.insert x i m, i)

  let ws = ["cat", "dog", "cat", "bird", "dog", "cat"]
  putStrLn $ "Слова: " ++ show ws
  putStrLn $ "ID:    " ++ show (labelUnique ws)

  -- Кодирование длин серий (RLE)
  let rleEncode :: Eq a => [a] -> [(Int, a)]
      rleEncode [] = []
      rleEncode (x:xs) =
        let (_, pairs) = mapAccumL step (1, x) xs
            step (cnt, prev) cur
              | cur == prev = ((cnt+1, prev), Nothing)
              | otherwise   = ((1, cur), Just (cnt, prev))
        in [(c, v) | Just (c,v) <- pairs] ++ [(fst $ fst (mapAccumL step (1, x) xs), x) | False] ++ []

  -- Упрощённый RLE
  let simpleRLE xs = map (\g -> (length g, head g)) $ groupBy' xs
      groupBy' [] = []
      groupBy' (x:xs) = let (same, rest) = span (== x) xs in (x:same) : groupBy' rest
  putStrLn $ "\nRLE \"aabbbccca\": " ++ show (simpleRLE "aabbbccca")

  -- mapAccumL на дереве
  putHeader "mapAccumL на TTree"
  let (_, labeledTree) = mapAccumL (\i x -> (i+1, (i::Int, x))) 0 numTree
  putStrLn $ "DFS-нумерация дерева: " ++ show (toList labeledTree)

demo_mapaccum2

Line 27: Short-circuited list comprehension
Found:
[(fst $ fst (mapAccumL step (1, x) xs), x) | False]
Why not:
[]Line 27: Evaluate
Found:
[(fst $ fst (mapAccumL step (1, x) xs), x) | False] ++ []
Why not:
([(fst $ fst (mapAccumL step (1, x) xs), x) | False])


=== mapAccumL — практические задачи ===
Слова: ["cat","dog","cat","bird","dog","cat"]
ID:    [0,1,0,2,1,0]

RLE "aabbbccca": [(2,'a'),(3,'b'),(3,'c'),(1,'a')]

=== mapAccumL на TTree ===
DFS-нумерация дерева: [(0,1),(1,3),(2,4),(3,5),(4,7),(5,8),(6,9)]

---
# 9️⃣ Продвинутые Паттерны

## 📐 `for`, `forM`, `for_` — флипнутые версии

```haskell
for  :: (Traversable t, Applicative f) => t a -> (a -> f b) -> f (t b)
for  = flip traverse

forM :: (Traversable t, Monad m) => t a -> (a -> m b) -> m (t b)
forM = flip mapM

for_ :: (Foldable t, Applicative f) => t a -> (a -> f b) -> f ()
for_ = flip traverse_
```

`for` удобнее когда структура данных — главный аргумент (не функция).

## 📐 Композиция `Traversable`

```haskell
-- Compose f g — вложенные Traversable
newtype Compose f g a = Compose { getCompose :: f (g a) }

instance (Traversable f, Traversable g) => Traversable (Compose f g) where
  traverse h (Compose fga) = Compose <$> traverse (traverse h) fga

-- Закон композиции traverse:
traverse (Compose . fmap g . f) = Compose . fmap (traverse g) . traverse f
```

## 📐 `Const` — `Traversable` как `Foldable`

```haskell
newtype Const m a = Const { getConst :: m }

instance Monoid m => Applicative (Const m) where
  pure _ = Const mempty
  Const f <*> Const x = Const (f <> x)

-- traverse (Const . f) = Const . foldMap f
-- Traversable → Foldable через Const!
```

In [27]:
-- Пример 1: for / forM — удобный синтаксис
demo_for :: IO ()
demo_for = do
  putHeader "for / forM"

  -- for — список действий с сбором результатов
  results <- for [1..5 :: Int] $ \n -> do
    return (n * n)
  putStrLn $ "Квадраты [1..5]: " ++ show results

  -- for с Maybe — валидация
  let validateAge :: Int -> Maybe Int
      validateAge n | n >= 0 && n <= 150 = Just n
                    | otherwise           = Nothing

  putStrLn $ "for validateAge [25, 30, 200]: " ++ show (for [25, 30, 200] validateAge)
  putStrLn $ "for validateAge [25, 30, 45]:  " ++ show (for [25, 30, 45] validateAge)

  -- for_ — только эффекты
  putStrLn "\nfor_ по Map:"
  let m = Map.fromList [("a", 1), ("b", 2), ("c", 3)] :: Map String Int
  for_ (Map.toList m) $ \(k, v) ->
    putStrLn $ "  " ++ k ++ " → " ++ show v

  -- forM для монадической нотации
  sums <- forM [[1,2,3],[4,5],[6,7,8,9]] $ \xs -> do
    let s = sum xs :: Int
    putStrLn $ "  sum " ++ show xs ++ " = " ++ show s
    return s
  putStrLn $ "Суммы подсписков: " ++ show sums

demo_for

Line 2: Use camelCase
Found:
demo_for :: IO ()
Why not:
demoFor :: IO ()Line 3: Use camelCase
Found:
demo_for = ...
Why not:
demoFor = ...


=== for / forM ===
Квадраты [1..5]: [1,4,9,16,25]
for validateAge [25, 30, 200]: Nothing
for validateAge [25, 30, 45]:  Just [25,30,45]

for_ по Map:
  a → 1
  b → 2
  c → 3
  sum [1,2,3] = 6
  sum [4,5] = 9
  sum [6,7,8,9] = 30
Суммы подсписков: [6,9,30]

In [28]:
-- Пример 2: Compose — вложенный traverse
newtype Compose f g a = Compose { getCompose :: f (g a) } deriving Show

instance (Functor f, Functor g) => Functor (Compose f g) where
  fmap h (Compose fga) = Compose (fmap (fmap h) fga)

instance (Foldable f, Foldable g) => Foldable (Compose f g) where
  foldMap h (Compose fga) = foldMap (foldMap h) fga

instance (Traversable f, Traversable g) => Traversable (Compose f g) where
  traverse h (Compose fga) = Compose <$> traverse (traverse h) fga

demo_compose :: IO ()
demo_compose = do
  putHeader "Compose — вложенные Traversable"

  -- Список списков
  let nested = Compose [[1,2,3],[4,5],[6,7,8,9]] :: Compose [] [] Int
  putStrLn $ "Вложенный список: " ++ show (getCompose nested)
  putStrLn $ "sum (Compose):    " ++ show (sum nested)
  putStrLn $ "toList (Compose): " ++ show (toList nested)

  -- traverse через Compose
  let doubled = fmap (*2) nested
  putStrLn $ "fmap (*2):        " ++ show (getCompose doubled)

  -- Maybe вложен в список
  let maybeNested = Compose [Just 1, Just 2, Just 3]
  let res1 = traverse (\x -> if x > 0 then Just (x*10) else Nothing) maybeNested
  putStrLn $ "\ntraverse (+pos check) [[Just 1,2,3]]: " ++ show (fmap getCompose res1)

  let maybeNested2 = Compose [Just 1, Nothing, Just 3]
  let res2 = traverse (\x -> if x > 0 then Just (x*10) else Nothing) maybeNested2
  putStrLn $ "traverse с Nothing: " ++ show (fmap getCompose res2)

demo_compose

Line 13: Use camelCase
Found:
demo_compose :: IO ()
Why not:
demoCompose :: IO ()Line 14: Use camelCase
Found:
demo_compose = ...
Why not:
demoCompose = ...


=== Compose — вложенные Traversable ===
Вложенный список: [[1,2,3],[4,5],[6,7,8,9]]
sum (Compose):    45
toList (Compose): [1,2,3,4,5,6,7,8,9]
fmap (*2):        [[2,4,6],[8,10],[12,14,16,18]]

traverse (+pos check) [[Just 1,2,3]]: Just [Just 10,Just 20,Just 30]
traverse с Nothing: Just [Just 10,Nothing,Just 30]

In [29]:
-- Пример 3: Traversable как Foldable через Const
-- Демонстрация: foldMap = traverse с Const

newtype Const' m a = Const' { getConst' :: m } deriving Show

instance Functor (Const' m) where
  fmap _ (Const' m) = Const' m

instance Monoid m => Applicative (Const' m) where
  pure _ = Const' mempty
  Const' f <*> Const' x = Const' (f <> x)

-- foldMap через traverse!
foldMapViaTraverse :: (Traversable t, Monoid m) => (a -> m) -> t a -> m
foldMapViaTraverse f = getConst' . traverse (Const' . f)

demo_const_traversal :: IO ()
demo_const_traversal = do
  putHeader "foldMap = traverse с Const"

  let xs = [3, 1, 4, 1, 5, 9, 2, 6] :: [Int]

  -- Через обычный foldMap
  let sumFM   = getSum $ foldMap Sum xs
  -- Через traverse с Const
  let sumConst = getSum $ foldMapViaTraverse Sum xs

  putStrLn $ "foldMap Sum:              " ++ show sumFM
  putStrLn $ "traverse (Const . Sum):   " ++ show sumConst
  putStrLn $ "Равны: " ++ show (sumFM == sumConst)

  -- На дереве
  putStrLn $ "\nfoldMap Sum numTree: " ++ show (getSum $ foldMap Sum numTree)
  putStrLn $ "foldMapVT Sum numTree: " ++ show (getSum $ foldMapViaTraverse Sum numTree)

demo_const_traversal

Line 17: Use camelCase
Found:
demo_const_traversal :: IO ()
Why not:
demoConstTraversal :: IO ()Line 18: Use camelCase
Found:
demo_const_traversal = ...
Why not:
demoConstTraversal = ...


=== foldMap = traverse с Const ===
foldMap Sum:              31
traverse (Const . Sum):   31
Равны: True

foldMap Sum numTree: 37
foldMapVT Sum numTree: 37

---
# 🔟 Производительность: Строгость, `Builder`, `Seq`

## 📐 Проблемы и решения

| Проблема | Причина | Решение |
|---------|---------|--------|
| `foldl` — утечка памяти | Накопление thunk-ов | `foldl'` или `foldMap'` |
| `foldr` — переполнение стека | Глубокая рекурсия | `foldl'` для конечных списков |
| `++` в `foldMap` — O(n²) | Конкатенация слева | `Builder` или `DList` |
| `length` — O(n) для `[]` | Подсчёт элементов | `Data.Sequence` — O(1) |
| `traverse` на `IO` | Последовательное выполнение | `async` / `concurrently` |

## ⭐ Difference Lists (DList)

```haskell
newtype DList a = DList { unDList :: [a] -> [a] }

instance Monoid (DList a) where
  mempty  = DList id
  DList f <> DList g = DList (f . g)

-- foldMap с DList = O(n) вместо O(n²)
toList t = unDList (foldMap (\x -> DList (x:)) t) []
```

In [30]:
-- Пример 1: DList — эффективная конкатенация
newtype DList a = DList { unDList :: [a] -> [a] }

instance Semigroup (DList a) where
  DList f <> DList g = DList (f . g)

instance Monoid (DList a) where
  mempty = DList id

singletonD :: a -> DList a
singletonD x = DList (x:)

fromDList :: DList a -> [a]
fromDList (DList f) = f []

toListFast :: Foldable t => t a -> [a]
toListFast = fromDList . foldMap singletonD

demo_dlist :: IO ()
demo_dlist = do
  putHeader "DList — O(n) конкатенация"

  let xss = [[1..20], [21..40], [41..60]] :: [[Int]]
  let r1 = foldr (++) [] xss
  let r3 = fromDList $ foldMap (DList . (++)) xss
  putStrLn $ "foldr (++): первые 5: " ++ show (take 5 r1)
  putStrLn $ "DList:      первые 5: " ++ show (take 5 r3)
  putStrLn $ "Равны: " ++ show (r1 == r3)

  -- Демонстрируем toListFast на дереве TTree
  putStrLn $ "\ntoListFast numTree: " ++ show (toListFast numTree)
  putStrLn $ "toList numTree:     " ++ show (toList numTree)
  putStrLn $ "Равны: " ++ show (toListFast numTree == toList numTree)

demo_dlist

Line 19: Use camelCase
Found:
demo_dlist :: IO ()
Why not:
demoDlist :: IO ()Line 20: Use camelCase
Found:
demo_dlist = ...
Why not:
demoDlist = ...Line 24: Use concat
Found:
foldr (++) []
Why not:
concat


=== DList — O(n) конкатенация ===
foldr (++): первые 5: [1,2,3,4,5]
DList:      первые 5: [1,2,3,4,5]
Равны: True

toListFast numTree: [1,3,4,5,7,8,9]
toList numTree:     [1,3,4,5,7,8,9]
Равны: True

In [31]:
-- Пример 2: foldMap' — строгий foldMap для агрегаций

demo_strict_foldmap :: IO ()
demo_strict_foldmap = do
  putHeader "foldMap' — строгий foldMap"

  let n = 10000 :: Int
  let xs = [1..n]

  let sumStrict = getSum $ foldMap' Sum xs
  let sumLazy   = getSum $ foldMap  Sum xs
  putStrLn $ "foldMap'  Sum [1..10000] = " ++ show sumStrict
  putStrLn $ "foldMap   Sum [1..10000] = " ++ show sumLazy

  -- Используем TTree из section 6
  putStrLn $ "\nfoldMap' Sum numTree = " ++ show (getSum $ foldMap' Sum numTree)
  putStrLn $ "length numTree = " ++ show (length numTree)

  -- Комплексная агрегация за один проход
  let data' = [1..10] :: [Int]
  let mySum   = getSum $ foldMap' Sum data'
  let myCount = getSum $ foldMap' (\_ -> Sum 1) data'
  let myMax   = maximum data'
  let myMin   = minimum data'
  putStrLn $ "\nАгрегация [1..10]:"
  putStrLn $ "  Сумма:    " ++ show (mySum :: Int)
  putStrLn $ "  Кол-во:   " ++ show (myCount :: Int)
  putStrLn $ "  Макс:     " ++ show myMax
  putStrLn $ "  Мин:      " ++ show myMin
  putStrLn $ "  Среднее:  " ++ show (fromIntegral mySum / fromIntegral myCount :: Double)

demo_strict_foldmap

Line 3: Use camelCase
Found:
demo_strict_foldmap :: IO ()
Why not:
demoStrictFoldmap :: IO ()Line 4: Use camelCase
Found:
demo_strict_foldmap = ...
Why not:
demoStrictFoldmap = ...Line 25: Redundant $
Found:
putStrLn $ "\nАгрегация [1..10]:"
Why not:
putStrLn "\nАгрегация [1..10]:"


=== foldMap' — строгий foldMap ===
foldMap'  Sum [1..10000] = 50005000
foldMap   Sum [1..10000] = 50005000

foldMap' Sum numTree = 37
length numTree = 7

Агрегация [1..10]:
  Сумма:    55
  Кол-во:   10
  Макс:     10
  Мин:      1
  Среднее:  5.5

In [32]:
-- Пример 3: Практический паттерн — одним проходом
-- Статистика за один проход через моноидный продукт

data Stats = Stats
  { sCount :: !Int
  , sSum   :: !Double
  , sMin   :: !Double
  , sMax   :: !Double
  } deriving Show

instance Semigroup Stats where
  Stats c1 s1 mn1 mx1 <> Stats c2 s2 mn2 mx2 =
    Stats (c1+c2) (s1+s2) (min mn1 mn2) (max mx1 mx2)

instance Monoid Stats where
  mempty = Stats 0 0 (1/0) (-1/0)  -- +inf, -inf

toStats :: Double -> Stats
toStats x = Stats 1 x x x

mean :: Stats -> Double
mean s = sSum s / fromIntegral (sCount s)

stddev :: [Double] -> Double
stddev xs = sqrt $ foldl' (\acc x -> acc + (x - m)^2) 0 xs / fromIntegral n
  where s = foldMap' toStats xs; m = mean s; n = sCount s

demo_stats :: IO ()
demo_stats = do
  putHeader "Статистика за один проход"
  let dataset = [2.5, 7.1, 3.8, 9.2, 1.4, 6.7, 4.3, 8.8, 5.6, 3.2] :: [Double]
  let stats = foldMap' toStats dataset
  putStrLn $ "Данные:       " ++ show dataset
  putStrLn $ "Count:        " ++ show (sCount stats)
  putStrLn $ "Sum:          " ++ show (sSum stats)
  putStrLn $ "Mean:         " ++ show (mean stats)
  putStrLn $ "Min:          " ++ show (sMin stats)
  putStrLn $ "Max:          " ++ show (sMax stats)
  putStrLn $ "Range:        " ++ show (sMax stats - sMin stats)
  putStrLn $ "Std Dev:      " ++ show (stddev dataset)

demo_stats

Line 28: Use camelCase
Found:
demo_stats :: IO ()
Why not:
demoStats :: IO ()Line 29: Use camelCase
Found:
demo_stats = ...
Why not:
demoStats = ...


=== Статистика за один проход ===
Данные:       [2.5,7.1,3.8,9.2,1.4,6.7,4.3,8.8,5.6,3.2]
Count:        10
Sum:          52.6
Mean:         5.26
Min:          1.4
Max:          9.2
Range:        7.799999999999999
Std Dev:      2.5188092424794695

---
# 1️⃣1️⃣ 🪞 Дуальность и Категорная Картина

## 📐 Теорема Макбрайда-Патерсона

**(McBride & Paterson, 2008)** — оригинальная статья «Applicative programming with effects».

Ключевые связи:

```
Functor ←── Foldable ─→ ???
  │              │
  └──── Traversable ──── требует Functor + Foldable
```

## 📐 Иерархия классов

```
         Functor
            │
      ┌─────┴─────┐
   Foldable   Applicative
      │           │
      └─────┬─────┘
       Traversable
```

**`Traversable` = `Functor` + `Foldable` + умение «поднимать» эффекты**

## 📐 `traverse` через `Representable`

Для `Representable` функторов (т.е. `t a ≅ r -> a`):
```haskell
traverse f ta = fmap (\k -> k (tabulate id)) $
                foldMap (\(i, fa) -> fmap (\b -> ...) fa) (indexed ta)
```

## 📐 Дуальность `Foldable` и `Unfoldable`

```
Foldable:   t a → (seed)  — «свернуть» структуру в значение
Unfoldable: (seed) → t a  — «развернуть» из значения (Ana-морфизм)

foldr   = катаморфизм (cata)
unfoldr = анаморфизм  (ana)
```

```haskell
-- Дуальность:
foldr  :: (a -> b -> b) -> b -> [a] -> b      -- cata
unfoldr :: (b -> Maybe (a, b)) -> b -> [a]   -- ana
```

In [33]:
import Data.List (unfoldr)

-- Дуальность foldr / unfoldr
demo_unfold :: IO ()
demo_unfold = do
  putHeader "foldr vs unfoldr — дуальность"

  -- unfoldr: seed → [a]
  let countdown = unfoldr (\n -> if n <= 0 then Nothing else Just (n, n-1)) 5
  putStrLn $ "unfoldr countdown 5: " ++ show countdown

  -- Числа Фибоначчи через unfoldr
  let fibs = take 10 $ unfoldr (\(a,b) -> Just (a, (b, a+b))) (0,1) :: [Int]
  putStrLn $ "Фибоначчи (10): " ++ show fibs

  -- Разложение числа на цифры
  let digits n = reverse $ unfoldr (\x -> if x == 0 then Nothing else Just (x `mod` 10, x `div` 10)) n
  putStrLn $ "digits 12345: " ++ show (digits (12345 :: Int))

  -- foldr сворачивает то, что unfoldr разворачивает
  let fromDigits = foldl' (\acc d -> acc * 10 + d) 0
  putStrLn $ "fromDigits (digits 12345): " ++ show (fromDigits (digits (12345 :: Int)))

  -- Бинарное представление через unfoldr
  let toBinary n = reverse $ unfoldr (\x -> if x == 0 then Nothing else Just (x `mod` 2, x `div` 2)) n
  putStrLn $ "\nBinary 42: " ++ concatMap show (toBinary (42 :: Int))
  putStrLn $ "Binary 255: " ++ concatMap show (toBinary (255 :: Int))

demo_unfold

Line 4: Use camelCase
Found:
demo_unfold :: IO ()
Why not:
demoUnfold :: IO ()Line 5: Use camelCase
Found:
demo_unfold = ...
Why not:
demoUnfold = ...


=== foldr vs unfoldr — дуальность ===
unfoldr countdown 5: [5,4,3,2,1]
Фибоначчи (10): [0,1,1,2,3,5,8,13,21,34]
digits 12345: [1,2,3,4,5]
fromDigits (digits 12345): 12345

Binary 42: 101010
Binary 255: 11111111

In [34]:
-- Связь Traversable и Applicative через закон натуральности

-- Натуральное преобразование сохраняет traverse:
-- t . traverse f = traverse (t . f)
-- (для натурального t :: f a -> g a)

demo_naturality :: IO ()
demo_naturality = do
  putHeader "Закон натуральности traverse"

  -- Пример: maybeToList - натуральное преобразование Maybe -> []
  let maybeToList :: Maybe a -> [a]
      maybeToList Nothing  = []
      maybeToList (Just x) = [x]

  let f :: Int -> Maybe Int
      f x = if even x then Just (x * 2) else Nothing

  let xs = [2, 4, 6, 1, 8] :: [Int]

  -- Левая сторона: t . traverse f
  let lhs = maybeToList (traverse f xs)
  -- Правая сторона: traverse (t . f)
  let rhs = traverse (maybeToList . f) xs

  putStrLn $ "t . traverse f = " ++ show lhs
  putStrLn $ "traverse (t.f) = " ++ show rhs
  putStrLn $ "Закон выполнен: " ++ show (lhs == rhs)

  -- Полная таблица классов
  putHeader "Иерархия Foldable/Traversable"
  putStrLn "  Functor → Applicative → Monad"
  putStrLn "     ↓"
  putStrLn "  Foldable"
  putStrLn "     ↓"
  putStrLn "  Traversable (требует Functor + Foldable)"
  putStrLn ""
  putStrLn "  traverse f  =  sequenceA . fmap f"
  putStrLn "  foldMap f   =  getConst . traverse (Const . f)"
  putStrLn "  fmap f      =  runIdentity . traverse (Identity . f)"

demo_naturality

Line 7: Use camelCase
Found:
demo_naturality :: IO ()
Why not:
demoNaturality :: IO ()Line 8: Use camelCase
Found:
demo_naturality = ...
Why not:
demoNaturality = ...


=== Закон натуральности traverse ===
t . traverse f = []
traverse (t.f) = []
Закон выполнен: True

=== Иерархия Foldable/Traversable ===
  Functor → Applicative → Monad
     ↓
  Foldable
     ↓
  Traversable (требует Functor + Foldable)

  traverse f  =  sequenceA . fmap f
  foldMap f   =  getConst . traverse (Const . f)
  fmap f      =  runIdentity . traverse (Identity . f)

In [35]:
-- Финальное демо: одна задача — три подхода
-- Задача: проверить список путей файловой системы,
-- заменить ~ на /home/user, проверить допустимость

type Path = String
type User = String

expandTilde :: User -> Path -> Path
expandTilde user ('~':'/':rest) = "/home/" ++ user ++ "/" ++ rest
expandTilde _ path = path

validatePath :: Path -> Either String Path
validatePath p
  | null p              = Left "Пустой путь"
  | head p /= '/'       = Left $ "Не абсолютный: " ++ p
  | ".." `isIn` p       = Left $ "Содержит ..: " ++ p
  | otherwise           = Right p
  where isIn sub str = any (sub `isPrefixOf`) (tails str)
        tails []     = [[]]
        tails s@(_:xs) = s : tails xs

paths :: [Path]
paths = ["~/docs/report.pdf", "~/code/Main.hs",
         "/etc/passwd", "relative/path",
         "~/../../etc/passwd",  -- атака!
         "~/music/song.mp3"]

demo_final :: IO ()
demo_final = do
  putHeader "Итоговое демо: обработка путей"

  -- Подход 1: map + filter (без Traversable)
  let expanded = map (expandTilde "alice") paths
  let valid1   = filter (\p -> case validatePath p of Right _ -> True; _ -> False) expanded
  putStrLn "\n1) map + filter:"
  mapM_ putStrLn valid1

  -- Подход 2: traverse Either (fail-fast)
  let result2 = traverse validatePath (map (expandTilde "alice") paths)
  putStrLn $ "\n2) traverse Either (fail-fast): " ++ show result2

  -- Подход 3: mapMaybe (все успешные)
  let valid3 = mapMaybe (\p -> case validatePath (expandTilde "alice" p) of
                                  Right r -> Just r
                                  Left _  -> Nothing) paths
  putStrLn "\n3) mapMaybe (все успешные):"
  mapM_ putStrLn valid3

  -- Подход 4: foldMap с логом ошибок
  let (errs, oks) = foldMap (\p ->
        case validatePath (expandTilde "alice" p) of
          Left e  -> ([e], [])
          Right r -> ([], [r])) paths
  putStrLn "\n4) foldMap — ошибки и успехи:"
  putStrLn $ "Ошибки: " ++ show errs
  putStrLn $ "Успехи: " ++ show oks

demo_final

Line 28: Use camelCase
Found:
demo_final :: IO ()
Why not:
demoFinal :: IO ()Line 29: Use camelCase
Found:
demo_final = ...
Why not:
demoFinal = ...Line 39: Fuse traverse/map
Found:
traverse validatePath (map (expandTilde "alice") paths)
Why not:
traverse (validatePath . expandTilde "alice") paths


=== Итоговое демо: обработка путей ===

1) map + filter:
/home/alice/docs/report.pdf
/home/alice/code/Main.hs
/etc/passwd
/home/alice/music/song.mp3

2) traverse Either (fail-fast): Left "\1053\1077 \1072\1073\1089\1086\1083\1102\1090\1085\1099\1081: relative/path"

3) mapMaybe (все успешные):
/home/alice/docs/report.pdf
/home/alice/code/Main.hs
/etc/passwd
/home/alice/music/song.mp3

4) foldMap — ошибки и успехи:
Ошибки: ["\1053\1077 \1072\1073\1089\1086\1083\1102\1090\1085\1099\1081: relative/path","\1057\1086\1076\1077\1088\1078\1080\1090 ..: /home/alice/../../etc/passwd"]
Успехи: ["/home/alice/docs/report.pdf","/home/alice/code/Main.hs","/etc/passwd","/home/alice/music/song.mp3"]

## 🧩Итоговая Категорная Картина

![Foldable vs Traversable](../diagrams/functors/ft_compare.svg)

**Ключевая интуиция:**
- `Foldable` позволяет «читать» элементы структуры, забывая о форме
- `Traversable` позволяет «трансформировать» элементы с эффектами, **помня** форму
- `traverse` сильнее `foldMap` (`foldMap = traverse c Const`) и `fmap` (`fmap = traverse c Identity`)

```
fmap f    = runIdentity . traverse (Identity . f)
foldMap f = getConst   . traverse (Const    . f)
```

## Категориальная Перспектива

Applicative = моноидальный функтор (Hask, x, 1):
- pure = моноидальная единица
- <*> = тензорное произведение

Traversable коммутирует с моноидальным:
```
t o f  -->  f o t    (через sequenceA)
```

**Законы Traversable:**
1. fmap phi . traverse f = traverse (phi . f)
2. traverse Identity = Identity
3. traverse (Compose . fmap g . f) = Compose . fmap (traverse g) . traverse f

**Схема классов:**
```
   Functor
     |    |
 Foldable Applicative
     |    |
  Traversable
```

fmap f = runIdentity . traverse (Identity . f)
foldMap f = getConst . traverse (Const . f)

![Foldable и Traversable](../diagrams/functors/ft_foldable.svg)

---

**← Предыдущий:** [Иерархия функторов](FunctorHierarchy.ipynb)  |  **Следующий →** [Монады](Monads.ipynb)
